In [234]:
# Imports and pip installations (if needed)
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import itertools
import warnings

warnings.filterwarnings("ignore", category=UserWarning) # Disable warnings for lack of convergence

# Part 1: Load the dataset

In [235]:
# Load the dataset (load remotely or locally)
iris_df = pd.read_csv('iris.csv')

# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
print(iris_df[:15])
print('\nsummary of dataframe:\n')
print(iris_df.describe())
print('\ninformation about label value counts: \n')
print(iris_df['species'].value_counts())

    sepal_length  sepal_width  petal_length  petal_width species
0            5.1          3.5           1.4          0.2  setosa
1            4.9          3.0           1.4          0.2  setosa
2            4.7          3.2           1.3          0.2  setosa
3            4.6          3.1           1.5          0.2  setosa
4            5.0          3.6           1.4          0.2  setosa
5            5.4          3.9           1.7          0.4  setosa
6            4.6          3.4           1.4          0.3  setosa
7            5.0          3.4           1.5          0.2  setosa
8            4.4          2.9           1.4          0.2  setosa
9            4.9          3.1           1.5          0.1  setosa
10           5.4          3.7           1.5          0.2  setosa
11           4.8          3.4           1.6          0.2  setosa
12           4.8          3.0           1.4          0.1  setosa
13           4.3          3.0           1.1          0.1  setosa
14           5.8         

## About the dataset
Explain what the data is in your own words. What are your features and labels? What is the mapping of your labels to the actual classes?

Features are the sepal length, sepal width, petal length and petal width (dimensions of two different anatomical parts of a flower), and the label is the species of the iris flower corresponding to the features. The labels, 'setosa', 'versicolor', and 'virginica' refer to three different species of iris - iris setosa, iris versicolor, and iris virginica. The label '0' maps to 'setosa', '1' maps to 'versicolor, and '2' maps to virginica.

# Part 2: Split the dataset into train and test

In [236]:
# Take the dataset and split it into our features (X) and label (y)
X = iris_df.drop(columns=['species'])
y = iris_df['species']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=16)

# Part 3: Logistic Regression

In [237]:
# i. Use sklearn to train a LogisticRegression model on the training set
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# ii. For a sample datapoint, predict the probabilities for each possible class
pred = model.predict(X_test)

# iii. Report on the score for Logistic regression model, what does the score measure?
scores = cross_val_score(LogisticRegression(max_iter=500), X, y, cv=10)
print(f'score: {scores.mean()}, std: {scores.std()}')

# iv. Extract the coefficents and intercepts for the boundary line(s)
print('coefficients:')
print(model.coef_)

print('\nintercept:')
print(model.intercept_)

score: 0.9733333333333334, std: 0.03265986323710904
coefficients:
[[-0.42096885  0.89873995 -2.42658362 -1.04675906]
 [ 0.40729905 -0.36902018 -0.11198827 -0.95607397]
 [ 0.0136698  -0.52971978  2.53857189  2.00283302]]

intercept:
[  9.59982992   2.62572021 -12.22555013]


## Logistic Regression Results
- The average score for the logistic regression model after 10 folds of cross validation is `0.973`, with a standard deviation of `0.032` - this means that it predicted `97%` of the test datapoints correctly on average
- The score measures the average percentage of test datapoints for which the logistic regression model makes the correct prediction of the datapoint's class based on the datapoint's features over 10 folds of cross validation

# Part 4: Support Vector Machine

In [238]:
# i. Use sklearn to train a Support Vector Classifier on the training set
clf = svm.SVC(probability=True)
clf.fit(X_train, y_train)

# ii. For a sample datapoint, predict the probabilities for each possible class
predicted_probs = clf.predict_proba([X_test.iloc[0]])
print(f'test datapoint: {np.array(X_test.iloc[0])}, class={y_test[0]} \npredicted probs for test datapoint: {predicted_probs[0]}')

# iii. Report on the score for the SVM, what does the score measure?
scores = cross_val_score(svm.SVC(probability=True), X, y, cv=10)
print(f'\n score of svm: {scores.mean()}, std: {scores.std()}')


test datapoint: [6.3 3.3 4.7 1.6], class=1 
predicted probs for test datapoint: [0.01022232 0.76256453 0.22721315]

 score of svm: 0.9733333333333334, std: 0.03265986323710904


## SVM Results
- The average score of the SVM over 10-fold cross validationis `0.973` with a standard deviation of `0.033` - this means that the svm predicted `97%` of the test datapoints correctly on average.
- The score measures the average percentage of test datapoints for which the trained SVM correctly predicted the datapoint's class given the datapoint's features as input over 10 folds of cross validation.

# Part 5: Neural Network

In [242]:
# i. Use sklearn to train a Neural Network (MLP Classifier) on the training set
clf = MLPClassifier(solver='adam', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=42)
clf.fit(X_train, y_train)

# ii. For a sample datapoint, predict the probabilities for each possible class
predicted_probs = clf.predict_proba(X_test[:1])
print(f'test datapoint: {np.array(X_test.iloc[0])}, class={y_test[0]} \npredicted probs for test datapoint: {predicted_probs[0]}')

# iii. Report on the score for the Neural Network, what does the score measure?
print(clf.score(X_test, y_test))

# iv: Experiment with different options for the neural network, report on your best configuration
archs = [(5, 3), (10, 3), (5, 5, 3), (5, 10, 10, 3), (10), (5), (3)]
activations = ['relu', 'logistic']
results = []
for arch, activation in list(itertools.product(archs, activations)):
    mlp = MLPClassifier(solver='adam', alpha=1e-5, max_iter=500, hidden_layer_sizes=arch, activation=activation)
    scores = cross_val_score(mlp, X, y, cv=10)
    print(f'architecture: {arch}, activation: {activation}, score: {scores.mean()}, cross-val-std: {scores.std()}')

test datapoint: [5.7 2.8 4.5 1.3], class=1 
predicted probs for test datapoint: [0.27142169 0.43379103 0.29478728]
0.7333333333333333
architecture: (5, 3), activation: relu, score: 0.7, cross-val-std: 0.16931233465600395
architecture: (5, 3), activation: logistic, score: 0.6466666666666666, cross-val-std: 0.030550504633038926
architecture: (10, 3), activation: relu, score: 0.7133333333333333, cross-val-std: 0.19333333333333338
architecture: (10, 3), activation: logistic, score: 0.7333333333333333, cross-val-std: 0.10749676997731401
architecture: (5, 5, 3), activation: relu, score: 0.6666666666666667, cross-val-std: 0.20439612955674522
architecture: (5, 5, 3), activation: logistic, score: 0.4666666666666666, cross-val-std: 0.1632993161855452
architecture: (5, 10, 10, 3), activation: relu, score: 0.7933333333333333, cross-val-std: 0.24846193538112296
architecture: (5, 10, 10, 3), activation: logistic, score: 0.33333333333333337, cross-val-std: 5.551115123125783e-17
architecture: 10, acti

## MLP Score Report / Training Results
- The score measures the percentage of test datapoints for which the MLP Classifier predicted the datapoint's class correctly given the datapoint's features as input. The score of the MLP was `0.8`, which means that the MLP predicted `80%` of the test datapoint classes correctly.
- After experimenting with multiple different architectures and activation functions using sklearn's `cross_val score` to evaluate models, I found the best configurations to be models with one hidden layer, and 10 neurons in that hidden layer. The best-performing model was the model with 10 neurons in the single hidden layer, and the logistic (sigmoid) activation function. This model had an average accuracy of 0.960 over 10 folds of cross validation, with a standard deviation of 0.053.

# Part 6: K-Nearest Neighbors

In [240]:
# i. Use sklearn to 'train' a k-Neighbors Classifier
# Note: KNN is a nonparametric model and technically doesn't require training
# fit will essentially load the data into the model see link below for more information
# https://stats.stackexchange.com/questions/349842/why-do-we-need-to-fit-a-k-nearest-neighbors-classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=32)
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train, y_train)

# ii. For a sample datapoint, predict the probabilities for each possible class
predicted_probs = neigh.predict_proba(X_test[:1])
print(f'test datapoint: {np.array(X_test.iloc[0])}, class={y_test[0]} \npredicted probs for test datapoint: {predicted_probs[0]}')

# iii. Report on the score for kNN, what does the score measure?
scores = cross_val_score(KNeighborsClassifier(n_neighbors=3), X, y, cv=10)
print(f'k: 3, mean score: {scores.mean()}, std: {scores.std()}')

test datapoint: [5.7 2.8 4.5 1.3], class=1 
predicted probs for test datapoint: [0. 1. 0.]
k: 3, mean score: 0.9666666666666666, std: 0.04472135954999579


## KNN Results
- The score measures the average percentage (over 10 folds of cross validation) of validation datapoints for which knn predicted the correct class for the datapoint given the datapoint's features as input. The score of KNN was `0.967` with a standard deviation of `0.045` - KNN predicted about `96 %` of test datapoints correctly on average.

# Part 7: Conclusions and takeaways

In your own words describe the results of the notebook. Which model(s) performed the best on the dataset? Why do you think that is? Did anything surprise you about the exercise?

Below are the final scores (accuracy) and standard deviations after 10 folds of cross validation for the models on the iris dataset: 

- Logistic regression:
    - score: `0.973`, standard deviation: `0.032`
- SVM:
    - score: `0.973`, standard deviation: `0.033`
- Neural Network (one 10-neuron hidden layer with sigmoid activation):
    - score: `0.960`, standard deviation: `0.053`
- K-nearest-neighbors
    - score: `0.967`, standard deviation: `0.045`

All models performed comparably well on this dataset, with logistic regression and SVM having the highest scores and near-equally low standard deviations over 10 folds of cross validation. I believe that this is because these models are the least difficult to tune - for logistic regression and SVM, I was able to use the sklearn's deafult parameters, and get decent results, whereas K-nearest neighbors requires the `k` parameter to be tuned (which I did not do), and the neural network has many different parameters that can be tuned (architecture, activation function, etc...), and the best-performing parameters I found in my limited experiment are probably not optimal.

One thing that surprised me about this experiment was how well logistic regression, SVM, and KNN performed on this dataset right out of the box. After I debugged the models, I got very high accuracy scores (close to 100% in many cases).